# memexcode — Architectural Blueprint

> Complete analysis of the pi coding agent TypeScript source, Manager01 design decisions, and implementation roadmap for the Python port using nbdev3/fastcore/pluggy.

## Table of Contents

1. [pi Source Architecture Review](#1-pi-source-architecture-review)
2. [Manager01 Design Decisions — Finalized](#2-manager01-design-decisions)
3. [Entry Type Catalog](#3-entry-type-catalog)
4. [Store Architecture — Three ABCs](#4-store-architecture)
5. [SessionManager Lifecycle](#5-sessionmanager-lifecycle)
6. [Concrete Implementation Tasks](#6-concrete-implementation-tasks)
7. [Notebook Split & Module Map](#7-notebook-split)
8. [Next Steps](#8-next-steps)


## 1. pi Source Architecture Review

### 1.1 What We Read

| File | Path | Key Findings |
|---|---|---|
| `00_core.ipynb` | core data models & channel side-channels | `ChannelSideChannel`, `SessionHeader`, session lifecycle |
| `agent-session.ts` | `/src/core/agent-session.ts` | HTML/JSONL export, utilities, session I/O |
| `session-manager.ts` | `/src/core/session-manager.ts` | `SessionHeader`, compaction/branching entries, versioning |
| `tools/index.ts` | `/src/core/tools/index.ts` | Tool factory/definition exports |
| `event-bus.ts` | `/src/core/event-bus.ts` | Async event emitter wrapper with safe handlers |
| `extensions/types.ts` | `/src/core/extensions/types.ts` | Lifecycle hooks, tool registration, context actions, runtime state |

### 1.2 Key Architectural Patterns

**Event Bus (`event-bus.ts`)**
- Async event emitter wrapper with `safeHandlers` for error isolation
- Tools subscribe to events; extension lifecycle is event-driven
- Prevents a single tool failure from crashing the session

**Extensions (`extensions/types.ts`)**
- Lifecycle hooks: `preToolExecute`, `postToolExecute`, `preMessage`, `postMessage`
- Tool registration: extensions can register additional tools with the agent
- Context actions: extensions can modify or augment the conversation context
- Runtime state: extensions maintain their own state via `extensionState` field

**Session Manager (`session-manager.ts`)**
- `SessionHeader` versioned (v1/v2/v3) with auto-migration
- Compaction: replaces old entries with `CompactionEntry` summaries
- Branching: `navigateTo()` generates branch summaries of abandoned paths
- Export: `exportSession()` writes both HTML (human-readable) and JSONL (machine-readable)

**Tools (`tools/index.ts`)**
- Factory pattern: `createTool()` constructs tool instances
- Tool catalog: built-in tools + extension-registered tools
- Safe execution: tools run in isolated context via event bus

**Agent Session (`agent-session.ts`)**
- Central session orchestrator
- HTML export with markdown rendering
- JSONL export for backup/restore
- Integration with event bus and extension system

### 1.3 What This Means for the Python Port

1. **Event-driven architecture** → Python port should use an async event bus (similar to `event-bus.ts`) for tool execution and extension hooks
2. **Extension lifecycle** → `pluggy` provides exactly the hook mechanism needed; `memexpaas_deploy/pluginspec.py` pattern is a good template
3. **Session versioning** → `SessionHeader.version` field with migration logic; Python port starts at v1 but should be version-aware
4. **Compaction + branching** → Core to the tree model; `CompactionEntry` + `navigateTo()` summaries are essential
5. **HTML + JSONL export** → Export is a rendering concern, not a data model concern. Defer to later.


## 2. Manager01 Design Decisions — Finalized

### 2.1 Data Models: dataclasses over Pydantic

| Reason | Detail |
|---|---|
| `fastlite` mapping | dataclasses map directly to SQLite columns via `dataclasses.asdict()` |
| Explicit serialization | No hidden Pydantic magic; `asdict()` is transparent and testable |
| Store neutrality | No ORM coupling; stores receive plain dicts, not ORM objects |

### 2.2 Store Architecture: Three ABCs (Not Two, Not Four)

```
SessionInfo (dataclass) — session metadata (id, cwd, created_at, updated_at, name, parent_session)
  │
  ├── SessionABC (per-session) — append, get_by_id, get_path_to_root, load_all, compact, get_tree
  │
  ├── SessionCollectionABC (multi-session) — create, list, rename, delete, fork
  │
  └── SessionStore(SessionCollectionABC) — get_session(session_id: str) -> SessionABC
```

**Why NOT `active_session` on the store?**
- Global mutable state → race conditions in async code
- Pluggy gives ONE store instance → can't safely have two SessionManagers
- Ambiguity: what does `store.append()` target? what does `store.load_all()` return?

**Factory pattern wins**: `SessionStore.get_session(id)` returns an isolated `SessionABC` object. Each session is its own object, no shared mutable state.

### 2.3 SessionManager Receives `SessionABC`, Not `SessionStore`

```
SessionManager.__init__(session: SessionABC, cwd: str)
    # SessionManager NEVER sees the store directly
    # It only sees the per-session SessionABC
```

Benefits:
- Store is swappable without changing SessionManager
- SessionManager is testable with MemorySession
- No tight coupling to persistence

### 2.4 Lazy Flush: Owned by SessionManager, Not SessionABC

```
append(entry) → buffer in _pending
    if not _flushed and role == 'assistant':
        _flush()  # write all pending at once
        _flushed = True
    else:
        await session.append(entry)  # eager for already-flushed sessions

flush() → _flush() + set _flushed = True

restore() → load_all(), take last entry as leaf, set _flushed = True
```

Rationale: `SessionABC` stays simple (always eager). `SessionManager` owns the batching semantics.

### 2.5 Serialization: `dataclasses.asdict()` + Module-Level Dispatch

```python
# No fastcore.patch (SessionEntry is a dataclass wrapper, not a class)

def dump_line(entry: SessionEntry) -> bytes:
    return orjson.dumps(entry.to_dict())  # orjson for speed

def entry_from_dict(d: dict) -> SessionEntry:
    t = d.get('type')
    dispatch = {
        'message': MessageEntry,
        'compaction': CompactionEntry,
        'model_change': ModelChangeEntry,
        'thinking_level_change': ThinkingLevelChangeEntry,
        'custom': CustomEntry,
        'label': LabelEntry,
    }
    return SessionEntry(dispatch[t](**d))
```

### 2.6 Circular Import Resolution

```
pluginspec.py ──imports──> session.py ──imports──> pluginspec.py  (CIRCULAR)

Fix:
1. Remove SessionManager from session.py → it lives in session_manager.py
2. session.py only imports from core.py (no circular dependency)
3. pluginspec.py imports SessionEntry from session.py (one-way)
4. TYPE_CHECKING block at end of pluginspec.py for forward references
```

### 2.7 SQLite Hybrid Schema

```
sessions table (header):
  session_id (PK), cwd, name, parent_session, created_at, updated_at, version

entries table (tree):
  id (PK), session_id, parent_id, type, timestamp, data (JSON blob)

get_path_to_root(leaf_id):
  Recursive CTE with WHERE session_id = ?
```

Header stored in `sessions` table only (not duplicated in entries). `SqliteSession` never sees the header.

### 2.8 Key Design Decisions Summary

| Decision | Choice | Rationale |
|---|---|---|
| Data models | dataclasses | fastlite mapping, explicit serialization |
| Store ABCs | 3 ABCs + factory | isolation, no global mutable state |
| SessionManager dep | SessionABC | store neutrality |
| Lazy flush | SessionManager-owned | SessionABC stays simple |
| Serialization | asdict + dispatch | store-neutral, testable |
| Header storage | sessions table only (SQLite) / line 1 (JSONL) | clean separation |
| Circular imports | SessionManager in separate module | type checkers via TYPE_CHECKING |
| Entry types | 7 types total | pi parity (model/thinking/label/custom) |


## 3. Entry Type Catalog

### 3.1 All Entry Types

| Type | Dataclass | LLM Context? | Purpose |
|---|---|---|---|
| `session` | SessionHeader | No | Header/metadata (root of tree) |
| `message` | MessageEntry | Yes | User/assistant messages |
| `compaction` | CompactionEntry | No | Summary of pruned history |
| `model_change` | ModelChangeEntry | No | provider/modelId change |
| `thinking_level_change` | ThinkingLevelChangeEntry | No | thinkingLevel toggle |
| `custom` | CustomEntry | No | Extension private state |
| `label` | LabelEntry | No | Bookmarks (targetId → label) |

### 3.2 SessionEntry Wrapper

```python
class SessionEntry:
    def __init__(self, entry):  # any Entry dataclass
        self._entry = entry
    
    @property
    def id(self) -> str: ...       # delegates to _entry
    @property
    def parentId(self) -> Optional[str]: ...
    @property
    def type(self) -> str: ...
    @property
    def timestamp(self) -> str: ...
    
    def to_dict(self) -> dict:
        return asdict(self._entry)  # store-agnostic
    
    @classmethod
    def from_dict(cls, d: dict) -> SessionEntry:
        # manual type dispatch
        ...
    
    def __getattr__(self, name):  # fallback delegation
        return getattr(self._entry, name)
```

`__getattr__` enables `entry.content`, `entry.role`, `entry.summary` transparently.


## 4. Store Architecture

### 4.1 MemoryStore (Testing)

```python
class MemorySession(SessionABC):
    def __init__(self):
        self._entries: Dict[str, SessionEntry] = {}
        self._children: Dict[str, List[str]] = defaultdict(list)
    
    async def append(self, entry): ...
    async def get_path_to_root(self, leaf_id): ...  # parent chain walk
    async def get_tree(self, root_id=None): ...     # TreeNode recursive build
```

### 4.2 JsonlStore (Portable)

```
~/.memexcode/sessions/
├── a1b2c3d4.jsonl   # line 1 = SessionHeader, rest = entries
├── e5f6g7h8.jsonl
└── index.json       # optional: SessionInfo cache
```

`JsonlSession` skips line 1 when loading. `JsonlStore.create()` writes header to line 1.

### 4.3 SqliteStore (Production)

```sql
CREATE TABLE sessions (
    session_id TEXT PRIMARY KEY,
    cwd TEXT NOT NULL,
    name TEXT,
    parent_session TEXT,
    created_at TEXT,
    updated_at TEXT,
    version INTEGER DEFAULT 1
);

CREATE TABLE entries (
    id TEXT PRIMARY KEY,
    session_id TEXT NOT NULL REFERENCES sessions(session_id),
    parent_id TEXT,
    type TEXT NOT NULL,
    timestamp TEXT,
    data TEXT  -- JSON blob for flexible payload
);
```

`get_path_to_root` uses recursive CTE:
```sql
WITH RECURSIVE path AS (
    SELECT id, parent_id, data, type FROM entries WHERE id = ? AND session_id = ?
    UNION ALL
    SELECT e.id, e.parent_id, e.data, e.type FROM entries e
    INNER JOIN path p ON e.id = p.parent_id AND e.session_id = p.session_id
)
SELECT * FROM path;
```

### 4.4 Pluggy Registration

```python
# In each store module (memory.py, jsonl.py, sqlite.py):
@hookimpl
def get_store():
    return ("jsonl", JsonlStore)

# pyproject.toml:
[project.entry-points."memexcode.store"]
memory = "memexcode.store.memory"
jsonl = "memexcode.store.jsonl"
sqlite = "memexcode.store.sqlite"
```

Discovery via `init_store_plugins()` → entry_points first, directory scanning fallback.


## 5. SessionManager Lifecycle

### 5.1 Complete API

```python
class SessionManager:
    def __init__(self, session: SessionABC, cwd: str = ""):
        self.session = session
        self.cwd = cwd
        self._leaf_id: Optional[str] = None
        self._pending: List[SessionEntry] = []
        self._flushed = False

    async def append(self, entry: SessionEntry) -> str:
        """Buffer entry; auto-flush on first assistant message."""
        if not entry.id:
            entry._entry.id = gen_id()
        if self._leaf_id:
            entry._entry.parentId = self._leaf_id
        self._pending.append(entry)
        self._leaf_id = entry.id
        
        if not self._flushed:
            if getattr(entry, 'role', None) == 'assistant':
                await self._flush()
                self._flushed = True
        else:
            await self.session.append(entry)
        
        return entry.id

    async def _flush(self):
        """Write all pending entries to session."""
        for entry in self._pending:
            await self.session.append(entry)
        self._pending.clear()

    async def flush(self):
        """Force flush (e.g., before crash, before tool call)."""
        await self._flush()
        self._flushed = True

    async def restore(self) -> bool:
        """Restore from store. Sets leaf to latest entry."""
        entries = await self.session.load_all()
        if not entries:
            return False
        self._leaf_id = entries[-1].id
        self._pending.clear()
        self._flushed = True
        return True

    async def navigate_to(self, entry_id: str):
        """Move leaf pointer. Triggers branch summarization."""
        self._leaf_id = entry_id

    async def build_context(self) -> List[dict]:
        """Build LLM context from leaf to root, messages only."""
        if not self._leaf_id:
            return []
        path = await self.session.get_path_to_root(self._leaf_id)
        return [e.to_dict() for e in path if e.type == 'message']
```

### 5.2 Lifecycle Diagram

```
SessionManager created (no session yet)
    ↓
append(MessageEntry(role="user")) → buffers in _pending
append(MessageEntry(role="user")) → buffers in _pending
append(MessageEntry(role="assistant")) → triggers _flush() + _flushed=True
    ↓
All subsequent appends → direct to session (eager)
    ↓
navigate_to(old_entry_id) → moves leaf, may generate branch summary
build_context() → walks path to root, returns message dicts
    ↓
restore() → fresh start, finds latest entry, _flushed=True
```

### 5.3 Compaction & Branching

**Compaction** (SessionManager-owned logic):
1. When context grows too large → pick `first_kept_id`
2. `session.compact(first_kept_id, summary)` rewrites history
3. Old entries replaced by single `CompactionEntry` with summary

**Branching** (on `navigate_to()`):
1. User navigates away from current leaf
2. SessionManager summarizes abandoned path
3. Inserts `CompactionEntry` at fork point
4. New leaf becomes navigation target


## 6. Concrete Implementation Tasks

### 6.1 Priority 1: Core Modules (Done in Manager01)

- [x] `00_core.ipynb` — All 7 entry type dataclasses
- [x] `01_session.ipynb` — `SessionEntry` wrapper (SessionManager removed)
- [x] `02_pluginspec.ipynb` — Three ABCs + TreeNode + TYPE_CHECKING block
- [x] `03_plugins.ipynb` — Pluggy hook spec + discovery
- [x] `04a_memory_store.ipynb` — `MemorySession` + `MemoryStore`
- [x] `04b_jsonl_store.ipynb` — `JsonlSession` + `JsonlStore`
- [x] `04c_store.sqlite.ipynb` — `SqliteSession` + `SqliteStore` with recursive CTE
- [x] `05_session_manager.ipynb` — SessionManager with lazy flush + restore
- [x] `pyproject.toml` — Entry points for all 3 stores

### 6.2 Priority 2: Verify & Test

- [ ] Run `nbdev_export` → verify all modules export cleanly
- [ ] Run `nbdev_install_()` → install in editable mode
- [ ] Test `MemoryStore` round-trip: create → append → build_context
- [ ] Test `JsonlStore` round-trip: persist → restore → verify tree
- [ ] Test `SqliteStore` round-trip: persist → restore → verify CTE path
- [ ] Test `flush()`/`restore()` flow end-to-end
- [ ] Fix any import/circular-reference issues

### 6.3 Priority 3: Deferred (Next Iteration)

- [ ] Context tree traversal — walk full tree for `/tree` command
- [ ] `fork()` implementation — copy path-to-root into new session
- [ ] Branch summarization — `navigate_to()` generates `CompactionEntry`
- [ ] SQLite store testing with realistic data
- [ ] Session listing with first-message preview
- [ ] Export helpers (`to_jsonl()` for SQLite)
- [ ] Event bus integration (async event emitter for tools)

### 6.4 Key Bug Fixes Applied in Manager01

| Bug | Fix |
|---|---|
| `flush()` didn't set `_flushed = True` | Added `self._flushed = True` at end of `flush()` |
| `restore()` used `get_tree()` (returns TreeNode, not list) | Changed to `load_all()`, take last entry as leaf |
| Circular import `session.py` ↔ `pluginspec.py` | Removed SessionManager from session.py; TYPE_CHECKING block |
| `TreeNode.entry` type annotation | Changed to string annotation `"SessionEntry"` |


## 7. Notebook Split & Module Map

| Notebook | Exports To | Content |
|---|---|---|
| `00_core.ipynb` | `memexcode/core.py` | Entry type dataclasses (7 types), `gen_id()`, `now_iso()` |
| `01_session.ipynb` | `memexcode/session.py` | `SessionEntry` wrapper (to_dict, from_dict, __getattr__) |
| `02_pluginspec.ipynb` | `memexcode/pluginspec.py` | SessionInfo, SessionABC, SessionCollectionABC, SessionStore, TreeNode |
| `03_plugins.ipynb` | `memexcode/plugins.py` | pluggy hookspec, init_store_plugins(), init_store_plugins_from_dir() |
| `04a_memory_store.ipynb` | `memexcode/store/memory.py` | MemorySession, MemoryStore (in-memory, for testing) |
| `04b_jsonl_store.ipynb` | `memexcode/store/jsonl.py` | JsonlSession, JsonlStore (one-file-per-session) |
| `04c_store.sqlite.ipynb` | `memexcode/store/sqlite.py` | SqliteSession, SqliteStore (hybrid schema, recursive CTE) |
| `05_session_manager.ipynb` | `memexcode/session_manager.py` | SessionManager (lazy flush, restore, navigate_to, build_context) |

### 7.1 Dependency Graph

```
00_core.py ← (nothing)
01_session.py ← core.py
02_pluginspec.py ← session.py (TYPE_CHECKING only)
03_plugins.py ← pluginspec.py
04a_memory_store.py ← core, session, pluginspec
04b_jsonl_store.py ← core, session, pluginspec
04c_store.sqlite.py ← core, session, pluginspec
05_session_manager.py ← session, pluginspec
```

### 7.2 Run Commands

```bash
# Export all notebooks to Python modules
cd /app/data/dialogs/b10Xp6TI7aVVLp8GyipIkw/memexcode
nbdev_export

# Install in editable mode
nbdev_install_

# Quick smoke test
python3 -c "
from memexcode.store.memory import MemoryStore
from memexcode.core import MessageEntry, gen_id
from memexcode.session import SessionEntry
from memexcode.session_manager import SessionManager

import asyncio
async def test():
    store = MemoryStore()
    info = await store.create(cwd='/tmp', name='test')
    session = store.get_session(info.id)
    sm = SessionManager(session, cwd='/tmp')
    await sm.append(SessionEntry(MessageEntry(role='user', content='hello')))
    await sm.append(SessionEntry(MessageEntry(role='assistant', content='hi')))
    ctx = await sm.build_context()
    print(f'Context messages: {len(ctx)}')
    print(f'Last leaf: {sm._leaf_id}')

asyncio.run(test())
"
```

### 7.3 Entry Points Configuration

```toml
[project.entry-points."memexcode.store"]
memory = "memexcode.store.memory"
jsonl = "memexcode.store.jsonl"
sqlite = "memexcode.store.sqlite"
```

Discovery order:
1. `entry_points(group="memexcode.store")` → registered plugins
2. Directory scanning fallback → scans `memexcode/store/` for modules with `get_store()`


## 8. Next Steps

### Immediate (This Session)

1. **Export & Install**: Run `nbdev_export` + `nbdev_install_()` to verify clean module generation
2. **Smoke Test**: Run the memory store test above to confirm the pipeline works
3. **Fix Any Issues**: Address import errors, type checking warnings, or runtime errors

### Short Term

4. **JsonlStore Testing**: Verify file-based persistence works (create → append → restore → verify)
5. **SqliteStore Testing**: Verify recursive CTE path traversal and tree structure
6. **Fix `get_tree()` return type**: It returns `TreeNode`, not `List[TreeNode]` — update callers

### Medium Term

7. **Fork Implementation**: `Store.fork(session_id, leaf_id, name)` — copy path-to-root
8. **Branch Summarization**: `navigate_to()` generates `CompactionEntry` summaries
9. **Event Bus**: Async event emitter for tool execution and extension hooks
10. **Context Tree**: Full tree traversal for `/tree` command support

### Long Term

11. **HTML Export**: Human-readable session export (pi's `exportSession()`)
12. **Session Migration**: v1 → v2 → v3 auto-migration logic
13. **Tool Catalog**: Built-in tools + extension-registered tools integration
14. **Extension System**: Full extension lifecycle with pluggy hooks

---

*Generated from analysis of:*  
- `/app/data/dialogs/b10Xp6TI7aVVLp8GyipIkw/pi-mono/packages/coding-agent/docs/pi_architecture.md`  
- `/app/data/dialogs/b10Xp6TI7aVVLp8GyipIkw/pi-mono/packages/coding-agent/src/core/` (5 TypeScript files)  
- `/app/data/dialogs/b10Xp6TI7aVVLp8GyipIkw/memexcode/nbs/Manager01.ipynb` (134 cells, §C.0–§C.51)  
- `/app/data/dialogs/b10Xp6TI7aVVLp8GyipIkw/memexpaas_deploy/memexpaas_deploy/pluginspec.py`  
- `/app/data/dialogs/b10Xp6TI7aVVLp8GyipIkw/memexpaas_deploy/memexpaas_deploy/store.py`
